# Model & Quality Monitors Plus Dashboard & Reports

## 1. Environment Setup

In [1]:
#1.1 Library Imports
import sagemaker
from sagemaker import Session
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker import image_uris, get_execution_role
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    ModelQualityMonitor,
    DatasetFormat,
    CronExpressionGenerator,
    EndpointInput,
)

import s3fs
import boto3
import pandas as pd
import numpy as np
import json
import time
from datetime import datetime, timedelta, timezone
from io import StringIO
from pathlib import Path

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
#1.2 Configuration

region = "us-east-1"
session = Session()
sm_client = boto3.client("sagemaker", region_name=region)
cw_client = boto3.client("cloudwatch", region_name=region)
s3_client = boto3.client("s3", region_name=region)

role = sagemaker.get_execution_role()
bucket = "sagemaker-us-east-1-418418308994"
prefix = "models/benchmarks"

## 2. Deploy Endpoint with Data Capture for Model Monitor

In [3]:
#2.1 Locate model artifacts
local_base = Path("/tmp/Models/benchmarks")
s3_base = f"s3://{bucket}/models/benchmarks"

xgb_paths = {
    "local_tar.gz": local_base / "xgboost/model.tar.gz",
    "s3_tar.gz": f"{s3_base}/xgboost/model.tar.gz",
}

model_data = (
    str(xgb_paths["local_tar.gz"])
    if xgb_paths["local_tar.gz"].exists()
    else xgb_paths["s3_tar.gz"]
)

In [4]:
#2.2 Container
xgboost_container = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
)

new_xgb_endpoint_name = f"xgb-benchmark-endpoint-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

In [5]:
#2.3 Model
xg_model = Model(
    image_uri=xgboost_container,
    model_data=model_data,
    role=role,
    sagemaker_session=session,
)

data_capture_prefix = f"{prefix}/datacapture"
data_capture_s3_uri = f"s3://{bucket}/{data_capture_prefix}"

# Deploy with data capture enabled
xg_predictor = xg_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=new_xgb_endpoint_name,
    data_capture_config=sagemaker.model_monitor.DataCaptureConfig(
        enable_capture=True,
        sampling_percentage=100,
        destination_s3_uri=data_capture_s3_uri,
        capture_options=["REQUEST", "RESPONSE"],
    ),
)

print("Endpoint:", new_xgb_endpoint_name)

------!Endpoint: xgb-benchmark-endpoint-20260211-061214


In [6]:
#2.4 Wait until endpoint is InService

print("Waiting for endpoint to be ready...")
while True:
    resp = sm_client.describe_endpoint(EndpointName=new_xgb_endpoint_name)
    status = resp["EndpointStatus"]
    print(" Status:", status)
    if status == "InService":
        print("Endpoint is ready!")
        break
    if status == "Failed":
        raise RuntimeError(f"Endpoint deployment failed: {resp.get('FailureReason')}")
    time.sleep(30)

Waiting for endpoint to be ready...
 Status: InService
Endpoint is ready!


## 3. Data Monitor & Data Quality

In [7]:
# 3.1: Create Data Quality Monitor
dq_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session,
)

In [8]:
# 3.2: Generate baseline using your data (creates stats/constraints)
baseline_dataset_uri = "s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv"
dq_baseline_uri = f"s3://{bucket}/{prefix}/monitoring/dq-baseline"

# Run baselining job
dq_baseline_job = dq_monitor.suggest_baseline(
    baseline_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=dq_baseline_uri,
    wait=True,
    logs=False,
)

print("✅ DQ baseline created!")

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-02-11-06-15-47-431


...........................................................!✅ DQ baseline created!


In [9]:
# 3.3: Get the generated stats/constraints URIs
try:
    dq_stats_uri = dq_monitor.latest_baselining_job.baseline_statistics.file_name
    dq_constraints_uri = dq_monitor.latest_baselining_job.suggested_constraints.file_name
except:
    # Fallback: find files manually
    s3 = boto3.client("s3")
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/monitoring/dq-baseline/")
    for obj in resp.get("Contents", []):
        if "statistics.json" in obj["Key"]:
            dq_stats_uri = f"s3://{bucket}/{obj['Key']}"
        if "constraints.json" in obj["Key"]:
            dq_constraints_uri = f"s3://{bucket}/{obj['Key']}"

print("Data Quality Stats:", dq_stats_uri)
print("Data Quality Constraints:", dq_constraints_uri)


Data Quality Stats: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/dq-baseline/statistics.json
Data Quality Constraints: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/dq-baseline/constraints.json


In [10]:
# 3.4: Create monitoring schedule using new baseline
schedule_name_xgb_dq = "xgb-data-quality-schedule"

try:
    dq_monitor.delete_monitoring_schedule(schedule_name_xgb_dq)
except:
    pass  # No existing schedule

dq_monitor.create_monitoring_schedule(
    monitor_schedule_name=schedule_name_xgb_dq,
    endpoint_input=new_xgb_endpoint_name,
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/data-quality/xgb",
    statistics=dq_stats_uri,
    constraints=dq_constraints_uri,
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print("✅ Data quality schedule live:", schedule_name_xgb_dq)


INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: xgb-data-quality-schedule


✅ Data quality schedule live: xgb-data-quality-schedule


In [12]:
dq_monitor.describe_schedule()

{'MonitoringScheduleArn': 'arn:aws:sagemaker:us-east-1:418418308994:monitoring-schedule/xgb-data-quality-schedule',
 'MonitoringScheduleName': 'xgb-data-quality-schedule',
 'MonitoringScheduleStatus': 'Scheduled',
 'MonitoringType': 'DataQuality',
 'CreationTime': datetime.datetime(2026, 2, 11, 6, 20, 50, 960000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 2, 11, 6, 20, 57, 288000, tzinfo=tzlocal()),
 'MonitoringScheduleConfig': {'ScheduleConfig': {'ScheduleExpression': 'cron(0 * ? * * *)'},
  'MonitoringJobDefinitionName': 'data-quality-job-definition-2026-02-11-06-20-50-044',
  'MonitoringType': 'DataQuality'},
 'EndpointName': 'xgb-benchmark-endpoint-20260211-061214',
 'LastMonitoringExecutionSummary': {'MonitoringScheduleName': 'xgb-data-quality-schedule',
  'ScheduledTime': datetime.datetime(2026, 2, 11, 6, 0, tzinfo=tzlocal()),
  'CreationTime': datetime.datetime(2026, 2, 11, 6, 0, 30, 329000, tzinfo=tzlocal()),
  'LastModifiedTime': datetime.datetime(2026, 2,

In [13]:
dq_executions = dq_monitor.list_executions()
dq_executions

## 4. Model Monitor Model Quality

In [23]:
#4.1 Create Model Quality Monitor
mq_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=session,
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [24]:
#4.2 Create a model quality baseline dataset

# Load data
baseline_df = pd.read_csv(f"s3://{bucket}/{prefix}/baseline_normalized.csv")
X_baseline = baseline_df.drop("target", axis=1)

# Get predictions from endpoint
csv_payload = X_baseline.to_csv(header=False, index=False)
predictor = Predictor(endpoint_name=new_xgb_endpoint_name, sagemaker_session=session)
response = predictor.predict(
    data=csv_payload,
    initial_args={"ContentType": "text/csv", "Accept": "text/csv"},
)

# Parse multi-class predictions
predictions_text = response.decode("utf-8").strip().split("\n")
prob_matrix = np.array([[float(x) for x in line.split(",") if x] for line in predictions_text])
pred_labels = np.argmax(prob_matrix, axis=1)  # argmax for multiclass

# Create baseline dataset
mq_baseline_df = pd.DataFrame({
    'prediction': pred_labels,
    'ground_truth_label': baseline_df['target'].values
})

mq_baseline_key = f"{prefix}/mq_baseline.csv"
csv_buffer = StringIO()
mq_baseline_df.to_csv(csv_buffer, index=False)
s3_client.put_object(
    Bucket=bucket,
    Key=mq_baseline_key,
    Body=csv_buffer.getvalue(),
    ContentType="text/csv",
)

mq_baseline_uri = f"s3://{bucket}/{mq_baseline_key}"
print("✅ MQ baseline uploaded:", mq_baseline_uri)

✅ MQ baseline uploaded: s3://sagemaker-us-east-1-418418308994/models/benchmarks/mq_baseline.csv


In [ ]:
#4.2 Run baselining job
mq_baseline_folder = f"s3://{bucket}/{prefix}/mq-baseline"

mq_monitor.suggest_baseline(
    baseline_dataset=mq_baseline_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/mq-baseline-simple",
    problem_type="MulticlassClassification",
    inference_attribute="prediction",
    ground_truth_attribute="ground_truth_label",
    wait=True,
    logs=True,
)

print("✅ Baselining job complete!")

In [33]:
#4.3 Get baseline files
job_desc = mq_monitor.latest_baselining_job.describe()
output_uri = job_desc['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri']
mq_stats_uri = f"{output_uri}/statistics.json"
mq_constraints_uri = f"{output_uri}/constraints.json"
print("✅ Stats:", mq_stats_uri)
print("✅ Constraints:", mq_constraints_uri)

✅ Stats: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/mq-baseline-simple/statistics.json
✅ Constraints: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/mq-baseline-simple/constraints.json


In [34]:
# Diagnose baselining failure
job_desc = mq_monitor.latest_baselining_job.describe()
print("Job Status:", job_desc["ProcessingJobStatus"])
print("Failure Reason:", job_desc.get("FailureReason", "None"))
print("Exit Message:", job_desc.get("ExitMessage", "None"))


Job Status: Completed
Failure Reason: None
Exit Message: Completed: Job completed successfully with no violations.


In [35]:
#4.4 Create a Model Quality Monitoring Schedule

mq_schedule_name = "xgb-model-quality-schedule"

mq_monitor.create_monitoring_schedule(
    monitor_schedule_name=mq_schedule_name,
    endpoint_input=EndpointInput(
        endpoint_name=new_xgb_endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
        inference_attribute="0",
    ),
    problem_type="MulticlassClassification",
    ground_truth_input=f"s3://{bucket}/{prefix}/ground-truth",
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/model-quality",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print("Model quality schedule:", mq_schedule_name)

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: xgb-model-quality-schedule


Model quality schedule: xgb-model-quality-schedule


In [36]:
#4.5 Create a Ground Truth File for Current Hour

s3_resource = boto3.resource("s3")

target_hour = datetime.utcnow().replace(minute=0, second=0, microsecond=0)
gt_key = f"{prefix}/ground-truth/{target_hour.strftime('%Y/%m/%d/%H')}/ground-truth.jsonl"

records = []
for i, gt_label in enumerate(baseline_df["target"].iloc[:100]):
    records.append(
        json.dumps(
            {
                "groundTruthData": {
                    "data": str(int(gt_label)),
                    "encoding": "CSV",
                },
                "eventMetadata": {
                    "eventId": f"gt-{i}",
                    "inferenceTime": target_hour.isoformat(),
                },
                "eventVersion": "0",
            }
        )
    )

s3_resource.Object(bucket, gt_key).put(Body="\n".join(records))

print("Ground truth uploaded to:", f"s3://{bucket}/{gt_key}")

Ground truth uploaded to: s3://sagemaker-us-east-1-418418308994/models/benchmarks/ground-truth/2026/02/11/06/ground-truth.jsonl


/tmp/ipykernel_9125/367159885.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  target_hour = datetime.utcnow().replace(minute=0, second=0, microsecond=0)


In [37]:
#4.6 Send predictions
test_data = baseline_df.drop("target", axis=1).iloc[:100]
predictor.predict(test_data.to_csv(header=False, index=False), 
                  initial_args={'ContentType': 'text/csv'})
print("✅ Predictions + ground truth ready!")

✅ Predictions + ground truth ready!


In [39]:
#4.8 Check monitor and executions
mq_monitor.describe_schedule()

{'MonitoringScheduleArn': 'arn:aws:sagemaker:us-east-1:418418308994:monitoring-schedule/xgb-model-quality-schedule',
 'MonitoringScheduleName': 'xgb-model-quality-schedule',
 'MonitoringScheduleStatus': 'Scheduled',
 'MonitoringType': 'ModelQuality',
 'CreationTime': datetime.datetime(2026, 2, 11, 6, 52, 7, 521000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 2, 11, 6, 52, 15, 254000, tzinfo=tzlocal()),
 'MonitoringScheduleConfig': {'ScheduleConfig': {'ScheduleExpression': 'cron(0 * ? * * *)'},
  'MonitoringJobDefinitionName': 'model-quality-job-definition-2026-02-11-06-52-06-840',
  'MonitoringType': 'ModelQuality'},
 'EndpointName': 'xgb-benchmark-endpoint-20260211-061214',
 'LastMonitoringExecutionSummary': {'MonitoringScheduleName': 'xgb-model-quality-schedule',
  'ScheduledTime': datetime.datetime(2026, 2, 10, 6, 0, tzinfo=tzlocal()),
  'CreationTime': datetime.datetime(2026, 2, 10, 6, 7, 59, 339000, tzinfo=tzlocal()),
  'LastModifiedTime': datetime.datetime(202

In [40]:
mq_executions = mq_monitor.list_executions()
mq_executions

## 4b Send Data to Endpoint

In [48]:
# Send Data
next_hour = (datetime.utcnow() + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)

# Predictions
predictor.predict(baseline_df.drop('target', axis=1).iloc[:200].to_csv(header=False, index=False), 
                  initial_args={'ContentType': 'text/csv'})

# Ground truth
s3_resource = boto3.resource('s3')
gt_key = f"{prefix}/ground-truth/{next_hour.strftime('%Y/%m/%d/%H')}/ground-truth.jsonl"

# Retrigger
for sched in ["xgb-data-quality-schedule", "xgb-model-quality-schedule"]:
    sm_client.start_monitoring_schedule(MonitoringScheduleName=sched)

print("✅ Data sent + retriggered for next hour")

/tmp/ipykernel_9125/971964714.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  next_hour = (datetime.utcnow() + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)


✅ Data sent + retriggered for next hour


## 5. Trigger Executions Manually

In [49]:
#5.1 Trigger manual run on existing schedules
schedules = ["xgb-data-quality-schedule", "xgb-model-quality-schedule"]

for schedule in schedules:
    try:
        sm_client.start_monitoring_schedule(MonitoringScheduleName=schedule)
        print(f"✅ {schedule} manual execution triggered!")
    except Exception as e:
        print(f"⚠️ {schedule}: {e}")

# Unified progress monitor
print("\n=== Monitoring Progress ===")
while True:
    all_done = True
    for schedule in schedules:
        resp = sm_client.describe_monitoring_schedule(MonitoringScheduleName=schedule)
        try:
            status = resp["LastMonitoringExecutionSummary"]["MonitoringExecutionStatus"]
            print(f"{schedule}: {status:<20}", end="")
        except KeyError:
            print(f"{schedule}: No execution yet    ", end="")
        
        if status not in ["Completed", "CompletedWithViolations", "Failed"]:
            all_done = False
    
    print()
    if all_done:
        break
    time.sleep(30)

print("✅ All executions complete!")


✅ xgb-data-quality-schedule manual execution triggered!
✅ xgb-model-quality-schedule manual execution triggered!

=== Monitoring Progress ===
xgb-data-quality-schedule: InProgress          xgb-model-quality-schedule: InProgress          
xgb-data-quality-schedule: InProgress          xgb-model-quality-schedule: InProgress          
xgb-data-quality-schedule: InProgress          xgb-model-quality-schedule: InProgress          
xgb-data-quality-schedule: InProgress          xgb-model-quality-schedule: InProgress          
xgb-data-quality-schedule: InProgress          xgb-model-quality-schedule: InProgress          
xgb-data-quality-schedule: InProgress          xgb-model-quality-schedule: InProgress          
xgb-data-quality-schedule: InProgress          xgb-model-quality-schedule: InProgress          
xgb-data-quality-schedule: InProgress          xgb-model-quality-schedule: InProgress          
xgb-data-quality-schedule: InProgress          xgb-model-quality-schedule: Failed         

In [53]:
#5.2 Check Results for Both Monitors
schedules = {
    "Data Quality": dq_monitor,
    "Model Quality": mq_monitor
}

for monitor_name, monitor_obj in schedules.items():
    print(f"\n=== {monitor_name} Results ===")
    
    # Wait for executions
    executions = []
    while not executions:
        executions = monitor_obj.list_executions()
        print(".", end="", flush=True)
        time.sleep(10)
    
    latest_execution = executions[-1]
    print()
    
    # Describe latest
    latest_job_desc = latest_execution.describe()
    print(f"Job Name: {latest_job_desc['ProcessingJobName']}")
    print(f"Status: {latest_job_desc['ProcessingJobStatus']}")
    print(f"Exit Message: {latest_job_desc.get('ExitMessage', 'None')}")
    print(f"Failure Reason: {latest_job_desc.get('FailureReason', 'None')}")
    
    # Output locations
    outputs = latest_job_desc["ProcessingOutputConfig"]["Outputs"]
    for output in outputs:
        s3_uri = output["S3Output"]["S3Uri"]
        print(f"Report Location: {s3_uri}")
    
    # Violation report (if exists)
    if "CompletedWithViolations" in latest_job_desc.get("ProcessingJobStatus", ""):
        print("📊 Violations detected - check S3 reports!")



=== Data Quality Results ===
.
Job Name: model-monitoring-202602110700-7205af54eee4eb167d657201
Status: Completed
Exit Message: CompletedWithViolations: Job completed successfully with 1 violations.
Failure Reason: None
Report Location: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/data-quality/xgb/xgb-benchmark-endpoint-20260211-061214/xgb-data-quality-schedule/2026/02/11/07

=== Model Quality Results ===
.
Job Name: groundtruth-merge-202602110700-421f72f21b6818737d32d66f
Status: Completed
Exit Message: None
Failure Reason: None
Report Location: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/model-quality/merge


In [59]:
#5.3 List EXACT folder contents to find files
s3 = boto3.client('s3')

# Data Quality folder
dq_folder = "models/benchmarks/monitoring/data-quality/xgb/xgb-benchmark-endpoint-20260211-061214/xgb-data-quality-schedule/2026/02/11/07/"
contents = s3.list_objects_v2(Bucket=bucket, Prefix=dq_folder)
files = [obj['Key'] for obj in contents.get('Contents', [])]
print("ALL files in DQ folder:")
for f in files:
    print(f"  {f}")

# Model Quality merge
mq_folder = "models/benchmarks/monitoring/model-quality/merge"
mq_contents = s3.list_objects_v2(Bucket=bucket, Prefix=mq_folder)
mq_files = [obj['Key'] for obj in mq_contents.get('Contents', [])]
print("\nALL files in MQ merge:")
for f in mq_files:
    print(f"  {f}")


ALL files in DQ folder:
  models/benchmarks/monitoring/data-quality/xgb/xgb-benchmark-endpoint-20260211-061214/xgb-data-quality-schedule/2026/02/11/07/constraint_violations.json

ALL files in MQ merge:


In [60]:
#5.4 Data Quality violations (actual file)
dq_key = "models/benchmarks/monitoring/data-quality/xgb/xgb-benchmark-endpoint-20260211-061214/xgb-data-quality-schedule/2026/02/11/07/constraint_violations.json"
obj = s3_client.get_object(Bucket=bucket, Key=dq_key)
violations = json.loads(obj["Body"].read())
print("📊 Data Quality Violations:")
print(json.dumps(violations, indent=2))


📊 Data Quality Violations:
{
  "violations": [
    {
      "feature_name": "Missing columns",
      "constraint_check_type": "missing_column_check",
      "description": "There are missing columns in current dataset. Number of columns in current dataset: 7, Number of columns in baseline constraints: 21"
    }
  ]
}


In [65]:
#5.5 Troubleshoot Model Quality Check
# 1. Send predictions FIRST
print("Step 1: Send predictions...")
predictor.predict(
    baseline_df.drop('target', axis=1).iloc[:200].to_csv(header=False, index=False),
    initial_args={'ContentType': 'text/csv'}
)
time.sleep(60)  # Data capture delay

# 2. Get EXACT prediction timestamp from data capture
prediction_hour = datetime.utcnow().replace(minute=0, second=0, microsecond=0)
print("Predictions timestamp:", prediction_hour)

# 3. Ground truth with EXACT SAME timestamp
s3_resource = boto3.resource('s3')
gt_key = f"{prefix}/ground-truth/{prediction_hour.strftime('%Y/%m/%d/%H')}/ground-truth.jsonl"

records = []
for i, gt_label in enumerate(baseline_df["target"].iloc[:200]):
    records.append(json.dumps({
        "groundTruthData": {"data": str(int(gt_label)), "encoding": "CSV"},
        "eventMetadata": {
            "eventId": f"gt-match-{i}",
            "inferenceTime": prediction_hour.isoformat()  # EXACT match!
        },
        "eventVersion": "0"
    }))

s3_resource.Object(bucket, gt_key).put(Body="\n".join(records))
print("✅ Ground truth matched to predictions")

# 4. Trigger
sm_client.start_monitoring_schedule(MonitoringScheduleName="xgb-model-quality-schedule")


Step 1: Send predictions...


/tmp/ipykernel_9125/795964115.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  prediction_hour = datetime.utcnow().replace(minute=0, second=0, microsecond=0)


Predictions timestamp: 2026-02-11 07:00:00
✅ Ground truth matched to predictions


{'ResponseMetadata': {'RequestId': '50b360e9-ba2a-4cb5-8aea-fa9256a09934',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '50b360e9-ba2a-4cb5-8aea-fa9256a09934',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'date': 'Wed, 11 Feb 2026 07:26:13 GMT',
   'content-length': '0'},
  'RetryAttempts': 0}}

In [67]:
#5.6 Wait for uploaded ground truths to hit
# Monitor MQ job
schedule_name = "xgb-model-quality-schedule"
while True:
    resp = sm_client.describe_monitoring_schedule(MonitoringScheduleName=schedule_name)
    status = resp["LastMonitoringExecutionSummary"]["MonitoringExecutionStatus"]
    print(f"MQ Status: {status}", end="\r")
    
    if status in ["Completed", "CompletedWithViolations", "Failed"]:
        print(f"\n🎉 MQ Final: {status}")
        break
    time.sleep(30)

MQ Status: Failed
🎉 MQ Final: Failed


In [68]:
import boto3
sm_client = boto3.client('sagemaker')

# Get LATEST MQ execution details
resp = sm_client.describe_monitoring_schedule(
    MonitoringScheduleName="xgb-model-quality-schedule"
)
latest_exec = resp["LastMonitoringExecutionSummary"]
print("Status:", latest_exec["MonitoringExecutionStatus"])
print("🚨 FAILURE REASON:", latest_exec["FailureReason"])
print("Processing Job:", latest_exec["ProcessingJobArn"])


Status: Failed
🚨 FAILURE REASON: Job inputs had no data
Processing Job: arn:aws:sagemaker:us-east-1:418418308994:processing-job/groundtruth-merge-202602110700-421f72f21b6818737d32d66f


## 6. Infrastructure Monitors (CloudWatch Alarms)

In [24]:
#Infrastructure monitoring alarms

endpoint_metric_dimensions = [
    {"Name": "EndpointName", "Value": new_xgb_endpoint_name},
]

# Latency alarm
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-High-Latency",
    AlarmDescription="Model latency above 10 seconds",
    Namespace="AWS/SageMaker",
    MetricName="ModelLatency",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Average",
    Period=300,
    EvaluationPeriods=2,
    Threshold=10000.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

# Invocation errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-Invocation-Errors",
    AlarmDescription="Invocation errors for endpoint",
    Namespace="AWS/SageMaker",
    MetricName="ModelInvocationErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=1,
    Threshold=5.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

# 5XX errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-5XX-Errors-High",
    AlarmDescription="5XX error rate high",
    Namespace="AWS/SageMaker",
    MetricName="Model5XXErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=2,
    Threshold=5.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

# 4XX errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-4XX-Errors-High",
    AlarmDescription="4XX error rate high",
    Namespace="AWS/SageMaker",
    MetricName="Model4XXErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=2,
    Threshold=10.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

print("Infrastructure alarms created.")


Infrastructure alarms created.


In [69]:
#6.2 Trigger alarm
cw_client = boto3.client('cloudwatch')

endpoint_name = "xgb-benchmark-endpoint-20260210-045052"  # Your endpoint
schedule_name = "xgb-model-quality-schedule"  # Your MQ schedule

# F2 Score Alarm (from your baseline)
cw_client.put_metric_alarm(
    AlarmName="XGB-F2-Score-Low",
    AlarmDescription="F2 score below baseline (drift detected)",
    ActionsEnabled=False,  # Enable=True for production
    MetricName="f2",  # Matches SageMaker ModelMonitor [web:62]
    Namespace="aws/sagemaker/Endpoints/model-metrics",
    Statistic="Average",
    Dimensions=[
        {"Name": "Endpoint", "Value": endpoint_name},
        {"Name": "MonitoringSchedule", "Value": schedule_name}
    ],
    Period=600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=0.625,  # Your low threshold for testing
    ComparisonOperator="LessThanOrEqualToThreshold",
    TreatMissingData="notBreaching"
)

print("✅ F2 Score alarm created!")


✅ F2 Score alarm created!


Validate on Sagemaker Interface

In [70]:
#More Alarms
metrics = [
    ("f1", 0.727, "F1 Score Low"),
    ("f2", 0.625, "F2 Score Low"), 
    ("accuracy", 0.940, "Accuracy Low"),
    ("auc", 0.940, "AUC Low"),
    ("precision", 1.0, "Precision Low"),
    ("recall", 0.571, "Recall Low")
]

for metric, threshold, name in metrics:
    cw_client.put_metric_alarm(
        AlarmName=f"XGB-{name}",
        AlarmDescription=f"{name.replace('-', ' ')} below baseline",
        MetricName=metric,
        Namespace="aws/sagemaker/Endpoints/model-metrics",
        Dimensions=[
            {"Name": "Endpoint", "Value": endpoint_name},
            {"Name": "MonitoringSchedule", "Value": schedule_name}
        ],
        Statistic="Average",
        Period=600,
        EvaluationPeriods=1,
        Threshold=threshold * 0.9,  # 90% of baseline
        ComparisonOperator="LessThanOrEqualToThreshold",
        TreatMissingData="notBreaching",
        ActionsEnabled=False
    )

## 7. CloudWatch Monitoring Dashboard

In [25]:
#7.1 CloudWatch Dashboard for ML endpoint

dashboard_name = "SageMaker-ML-Benchmarks"

dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0,
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Endpoint – Invocations & Latency",
                "metrics": [
                    ["AWS/SageMaker", "Invocations", "EndpointName", new_xgb_endpoint_name],
                    [".", "ModelLatency", ".", "."],
                ],
                "stacked": False,
                "stat": "Average",
                "period": 60,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Endpoint – Errors",
                "metrics": [
                    ["AWS/SageMaker", "ModelInvocationErrors", "EndpointName", new_xgb_endpoint_name],
                    [".", "Invocation4XXErrors", ".", "."],
                    [".", "Invocation5XXErrors", ".", "."],
                ],
                "stacked": False,
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Data Quality Violations",
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "DataQualityViolation",
                        "MonitoringSchedule",
                        schedule_name_xgb_dq,
                    ]
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Model Quality Violations",
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "ModelQualityViolation",
                        "MonitoringSchedule",
                        "xgb-model-quality-schedule",
                    ]
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 12,
            "width": 24,
            "height": 6,
            "properties": {
                "title": "Alarm States",
                "metrics": [
                    ["AWS/CloudWatch", "AlarmState", "AlarmName", "XGB-Endpoint-High-Latency"],
                    ["...", "XGB-Endpoint-Invocation-Errors"],
                    ["...", "XGB-Endpoint-5XX-Errors-High"],
                    ["...", "XGB-Endpoint-4XX-Errors-High"],
                ],
                "stat": "Maximum",
                "period": 300,
                "region": region,
            },
        },
    ]
}

cw_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard_body),
)

print("Dashboard created:", dashboard_name)
print(
    f"URL: https://console.aws.amazon.com/cloudwatch/home?region={region}"
    f"#dashboards:name={dashboard_name}"
)


Dashboard created: SageMaker-ML-Benchmarks
URL: https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=SageMaker-ML-Benchmarks


## 8. Verification

In [29]:
#Verify data capture and ground truth ready

fs = s3fs.S3FileSystem()
now = datetime.now(timezone.utc)
current_hour = now.replace(minute=0, second=0, microsecond=0)

dc_path = f"{bucket}/{prefix}/datacapture/{new_xgb_endpoint_name}/AllTraffic/{current_hour.strftime('%Y/%m/%d/%H')}/"
gt_path = f"{bucket}/{prefix}/ground-truth/{current_hour.strftime('%Y/%m/%d/%H')}/"

print("DataCapture:", "✅" if len(fs.ls(dc_path, detail=True)) > 0 else "❌ Wait/re-send")
print("GroundTruth:", "✅" if len(fs.ls(gt_path, detail=True)) > 0 else "❌ Upload more")

DataCapture: ✅
GroundTruth: ✅


In [52]:
def check_monitor_status():
    schedules = [schedule_name_xgb_dq, "xgb-model-quality-schedule"]
    print("=== Monitor Status ===")
    for sched in schedules:
        try:
            resp = sm_client.describe_monitoring_schedule(MonitoringScheduleName=sched)
            exec_summary = resp.get("LastMonitoringExecutionSummary", {})
            status = exec_summary.get("MonitoringExecutionStatus", "No execution")
            time = exec_summary.get("ScheduledTime", "N/A")
            print(f"{sched:25}: {status} ({time})")
        except Exception as e:
            print(f"{sched:25}: Error - {e}")
    print("====================")

check_monitor_status()

=== Monitor Status ===
xgb-data-quality-schedule: CompletedWithViolations (2026-02-11 07:00:00+00:00)
xgb-model-quality-schedule: Failed (2026-02-11 07:00:00+00:00)


In [32]:
#Full MQ schedule diagnosis
mq_schedule_name = "xgb-model-quality-schedule"

resp = sm_client.describe_monitoring_schedule(MonitoringScheduleName=mq_schedule_name)
print("Schedule Status:", resp["MonitoringScheduleStatus"])
print("Last Execution:", resp.get("LastMonitoringExecutionSummary", "None"))

# List recent executions
executions = sm_client.list_monitoring_executions(
    MonitoringScheduleName=mq_schedule_name,
    MaxResults=5,
    SortOrder="Descending"
)
print("\nRecent Executions:")
for exec in executions["MonitoringExecutionSummaries"]:
    print(f"  {exec['ScheduledTime']}: {exec['MonitoringExecutionStatus']}")



Schedule Status: Scheduled
Last Execution: {'MonitoringScheduleName': 'xgb-model-quality-schedule', 'ScheduledTime': datetime.datetime(2026, 2, 10, 6, 0, tzinfo=tzlocal()), 'CreationTime': datetime.datetime(2026, 2, 10, 6, 7, 59, 339000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2026, 2, 10, 6, 17, 7, 597000, tzinfo=tzlocal()), 'MonitoringExecutionStatus': 'Failed', 'ProcessingJobArn': 'arn:aws:sagemaker:us-east-1:418418308994:processing-job/groundtruth-merge-202602100600-d081f80f835cf867bcd3661c', 'EndpointName': 'xgb-benchmark-endpoint-20260210-045052', 'FailureReason': 'Job inputs had no data'}

Recent Executions:
  2026-02-10 06:00:00+00:00: Failed
  2026-02-09 08:00:00+00:00: Failed
  2026-02-09 07:00:00+00:00: Failed
  2026-02-09 06:00:00+00:00: Failed
  2026-02-09 05:00:00+00:00: Failed


## 9. Generate Model & Data Reports

This is for after the monitor runs.

In [ ]:
#Helper to inspect latest executions and get report locations

def get_latest_execution(schedule_name):
    resp = sm_client.list_monitoring_executions(
        MonitoringScheduleName=schedule_name,
        MaxResults=5,
        SortOrder="Descending",
    )
    if not resp["MonitoringExecutionSummaries"]:
        print("No executions found for", schedule_name)
        return None
    return resp["MonitoringExecutionSummaries"][0]

for name in [schedule_name_xgb_dq, "xgb-model-quality-schedule"]:
    latest = get_latest_execution(name)
    if latest:
        print("\nSchedule:", name)
        print(" Status:", latest["MonitoringExecutionStatus"])
        print(" ScheduledTime:", latest["ScheduledTime"])


In [ ]:
#Example: download latest data quality report artifacts

dq_latest = get_latest_execution(schedule_name_xgb_dq)
if dq_latest and "ProcessingJobArn" in dq_latest:
    job_name = dq_latest["ProcessingJobArn"].split("/")[-1]
    job_desc = sm_client.describe_processing_job(ProcessingJobName=job_name)
    outputs = job_desc["ProcessingOutputConfig"]["Outputs"]
    for out in outputs:
        uri = out["S3Output"]["S3Uri"]
        print("DQ output:", uri)
        # Typically contains /constraints.json and /statistics.json


In [ ]:
#Example: download latest model quality report artifacts

mq_latest = get_latest_execution("xgb-model-quality-schedule")
if mq_latest and "ProcessingJobArn" in mq_latest:
    job_name = mq_latest["ProcessingJobArn"].split("/")[-1]
    job_desc = sm_client.describe_processing_job(ProcessingJobName=job_name)
    outputs = job_desc["ProcessingOutputConfig"]["Outputs"]
    for out in outputs:
        uri = out["S3Output"]["S3Uri"]
        print("MQ output:", uri)


## 10. Clean Up Resources

In [41]:
# Delete all monitoring schedules
schedules = [
    "xgb-data-quality-schedule",
    "xgb-model-quality-schedule",
    "xgb-model-quality-schedule",
    "xgb-model-quality-schedule"
]

for schedule in schedules:
    try:
        sm_client.delete_monitoring_schedule(MonitoringScheduleName=schedule)
        print(f"✅ Deleted schedule: {schedule}")
    except Exception as e:
        print(f"⚠️ {schedule}: {e}")

⚠️ xgb-data-quality-schedule: An error occurred (ResourceNotFound) when calling the DeleteMonitoringSchedule operation: Monitoring Schedule arn:aws:sagemaker:us-east-1:418418308994:monitoring-schedule/xgb-data-quality-schedule not found
⚠️ xgb-model-quality-schedule: An error occurred (ResourceNotFound) when calling the DeleteMonitoringSchedule operation: Monitoring Schedule arn:aws:sagemaker:us-east-1:418418308994:monitoring-schedule/xgb-model-quality-schedule not found
⚠️ xgb-model-quality-schedule: An error occurred (ResourceNotFound) when calling the DeleteMonitoringSchedule operation: Monitoring Schedule arn:aws:sagemaker:us-east-1:418418308994:monitoring-schedule/xgb-model-quality-schedule not found
⚠️ xgb-model-quality-schedule: An error occurred (ResourceNotFound) when calling the DeleteMonitoringSchedule operation: Monitoring Schedule arn:aws:sagemaker:us-east-1:418418308994:monitoring-schedule/xgb-model-quality-schedule not found


In [40]:
# Delete endpoint (stops data capture)
try:
    sm_client.delete_endpoint(EndpointName=new_xgb_endpoint_name)
    print(f"✅ Deleting endpoint: {new_xgb_endpoint_name}")
except Exception as e:
    print(f"⚠️ Endpoint delete: {e}")


✅ Deleting endpoint: xgb-benchmark-endpoint-20260211-052400


In [2]:
import boto3
sm_client = boto3.client('sagemaker', region_name='us-east-1')
cw_client = boto3.client('cloudwatch', region_name='us-east-1')

print("=== CLEANUP STATUS ===")
eps = sm_client.list_endpoints()["Endpoints"]
print("Endpoints:", [e["EndpointName"] for e in eps] or "✅ NONE")

schedules = sm_client.list_monitoring_schedules()["MonitoringScheduleSummaries"]
print("Schedules:", [s["MonitoringScheduleName"] for s in schedules] or "✅ NONE")

alarms = cw_client.describe_alarms(AlarmNamePrefix="XGB-")["MetricAlarms"]
print("XGB Alarms:", [a["AlarmName"] for a in alarms] or "✅ NONE")

print("✅ Ready for restart!")


=== CLEANUP STATUS ===
Endpoints: ['xgb-benchmark-endpoint-20260210-045052', 'xgb-benchmark-endpoint-20260209-062303', 'xgb-benchmark-endpoint-20260209-023949']
Schedules: ✅ NONE
XGB Alarms: ['XGB-Endpoint-4XX-Errors-High', 'XGB-Endpoint-5XX-Errors-High', 'XGB-Endpoint-High-Latency', 'XGB-Endpoint-Invocation-Errors']
✅ Ready for restart!
